# ATM Straddle Backtest

This notebook runs the backtest for the `ATMStraddleStrategy` and reports the results.

**Strategy**: At every second, find the strike nearest to the front-month futures price. Hold a long straddle (CE + PE) at that strike. Roll when the ATM strike changes. Force-close at 15:29:59.

**Universe**: NIFTY + BANKNIFTY, November 2022 (21 trading days).

Plots are saved to `results/`.

In [ ]:
from pathlib import Path

import stratbt.metrics as metrics
from stratbt import BacktestEngine, ExecutionModel, NSETickProvider
from stratbt.strategy.atm_straddle import ATMStraddleStrategy

DATA_DIR    = Path("../data/allData")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Run the backtest

The `strategy_factory` argument is the only thing that changes between strategies — swap it out to test a different one.

In [ ]:
provider = NSETickProvider(DATA_DIR)
universe = provider.universe           # strikes/expiries index, used by later cells
# ExecutionModel.instant() = same-second fills, zero costs: the original (optimistic) model.
# Drop it (or use ExecutionModel.realistic()) for next-second fills and trading costs.
engine   = BacktestEngine(provider, ATMStraddleStrategy, execution=ExecutionModel.instant())
result   = engine.run_all(underliers=["NIFTY", "BANKNIFTY"])

## 2. Summary table

One row per underlier, all key metrics.

In [ ]:
metrics.summary_table(result)

### Reading the summary table

**Total PnL**: BANKNIFTY lost 1507 index points, NIFTY lost 508. All in index points, multiplier = 1 (see assumptions).

**Sharpe / Sortino / Calmar**: All deeply negative for both underliers. These are not meaningful as standalone signals with only 21 days — stated here for completeness with the caveat that N=21 makes all risk-adjusted metrics statistically unreliable estimates.

**Max drawdown duration = 19/21**: Both underliers were in drawdown for 19 of 21 trading days. The strategy briefly reached its peak on Nov 2 and never recovered from that point forward.

**Win rate**: BANKNIFTY 9.5%, NIFTY 19%. Out of 21 days, BANKNIFTY was profitable on 2 days and NIFTY on 4. This is characteristic of a long straddle in a low-volatility regime — theta bleeds you most days and you only win on high-movement days.

**Profit factor**: 0.03 and 0.17. For every ₹1 made on winning days, BANKNIFTY lost ₹33 and NIFTY lost ₹6 on losing days.

**Avg holding time**: ~111-120 seconds. The average position was held for under 2 minutes before being rolled. On days with heavy ping-pong (adjacent strikes), many individual straddle holds lasted just 1-2 seconds.

**Total trades**: 32,692 across the month. At the lot size of 1 (which is not a realistic lot — NIFTY lot = 50, BANKNIFTY lot = 25 in November 2022), this is the raw transaction count. See cost sensitivity section.

## 3. Daily PnL table

In [ ]:
result.daily_pnl

## 4. Cumulative PnL

In [ ]:
fig = metrics.plot_cumulative_pnl(result)
fig.savefig(RESULTS_DIR / "01_cumulative_pnl.png", dpi=150, bbox_inches="tight")

**What:** Cumulative sum of daily PnL from Nov 1 to Nov 30.

**Observation:** BANKNIFTY (blue) deteriorates faster and more deeply than NIFTY (orange). Both only had positive cumulative PnL on Nov 2, before entering a continuous losing streak. The combined portfolio (black dashed) finished at -2015 index points.

## 5. Daily PnL bars

In [ ]:
fig = metrics.plot_daily_pnl(result)
fig.savefig(RESULTS_DIR / "02_daily_pnl.png", dpi=150, bbox_inches="tight")

**What:** Daily profit and loss for each underlier, shown side by side.

**Observation:** Nov 3 (expiry day for BANKNIFTY front week) was by far the worst day — BANKNIFTY lost 257 points in a single session. This is not coincidental: on expiry day, options premium collapses rapidly toward intrinsic value. The straddle you're long loses value quickly as expiry approaches and the market settles. The two best days (Nov 2 and Nov 9) were days where the underlying made a large sustained directional move.

## 6. Drawdown

In [ ]:
fig = metrics.plot_drawdown(result)
fig.savefig(RESULTS_DIR / "03_drawdown.png", dpi=150, bbox_inches="tight")

**What:** How far below the running equity peak you are at each point. Starts at 0 and goes negative whenever cumulative PnL falls below its previous high.

**Observation:** Neither line returns to 0 after Nov 2. Max drawdown duration = 19 days for both — the strategy was effectively in continuous drawdown for the entire month after the first week. BANKNIFTY's max drawdown was 1495 points, NIFTY's was 533.

## 7. Intraday MTM — sample days

Per-second MTM for one BANKNIFTY day and one NIFTY day.

In [ ]:
fig = metrics.plot_intraday_mtm(result, trading_date="2022-11-01", underlier="BANKNIFTY")
fig.savefig(RESULTS_DIR / "04_intraday_mtm_banknifty.png", dpi=150, bbox_inches="tight")

**What:** Mark-to-market equity at every second of the Nov 1 session for BANKNIFTY.

**Observation:** The strategy was negative almost all day (-70 to -100 range), with one brief recovery spike near 12:30 PM where it nearly reached breakeven. The spike corresponds to a large fast move in BANKNIFTY that momentarily made the long straddle profitable. But the market quickly calmed, and the strategy ended at -58.

In [ ]:
fig = metrics.plot_intraday_mtm(result, trading_date="2022-11-01", underlier="NIFTY")
fig.savefig(RESULTS_DIR / "05_intraday_mtm_nifty.png", dpi=150, bbox_inches="tight")

**What:** Mark-to-market equity at every second of the Nov 1 session for NIFTY.

**Observation:** Unlike BANKNIFTY, NIFTY spent most of the day IN PROFIT (green shading). It peaked near +26 in early afternoon, holding positive for roughly 5 hours. The final 30 minutes wiped out all gains, and the day closed at -2.05.

## 8. Rolls per day

How many times the strategy rolled the ATM strike each day. A roll = the futures price crossed into a new nearest-strike bucket while the position was open.

In [ ]:
print(metrics.rolls_per_day(result).groupby("underlier")[["rolls", "trades"]].describe())

fig = metrics.plot_rolls_per_day(result)
fig.savefig(RESULTS_DIR / "06_rolls_per_day.png", dpi=150, bbox_inches="tight")

## 9. Strike held — sample day

Shows the ATM strike held at every second of a single trading day — makes the rolling behaviour visually obvious.

In [ ]:
fig = metrics.plot_holdings(result, trading_date="2022-11-01", underlier="BANKNIFTY")
fig.savefig(RESULTS_DIR / "07_strike_held_banknifty.png", dpi=150, bbox_inches="tight")

**What:** Which strike the strategy was holding at every second on Nov 1 for BANKNIFTY.

**Observation:** The strike trended downward from ~41600 in the morning to ~41500 in the afternoon, reflecting BANKNIFTY's intraday decline. The rapid vertical oscillations ("ping-pong") appear wherever the futures price hovered near the midpoint between two adjacent strikes — the strategy rolled back and forth many times within a few minutes.

In [ ]:
fig = metrics.plot_holdings(result, trading_date="2022-11-01", underlier="NIFTY")
fig.savefig(RESULTS_DIR / "08_strike_held_nifty.png", dpi=150, bbox_inches="tight")

**What:** Which strike the strategy held at every second on Nov 1 for NIFTY.

**Observation:** Less directional drift than BANKNIFTY. The large dip to 18100 near 13:00 corresponds to a sharp intraday drop in NIFTY before it recovered. The denser oscillations in the afternoon show the ping-pong behaviour.

## 10. Transaction cost sensitivity

The strategy rolls frequently (~hundreds of times per day). Even small per-trade costs have a large cumulative impact. This table shows gross PnL vs net PnL at different cost assumptions.

In [ ]:
metrics.cost_sensitivity(result)

In [ ]:
fig = metrics.plot_cost_sensitivity(result)
fig.savefig(RESULTS_DIR / "09_cost_sensitivity.png", dpi=150, bbox_inches="tight")

**What:** Net PnL at different per-transaction cost assumptions, in basis points of the option premium.

**Observation:** Gross PnL (0 bps) = -2015. At just 1 bps: -2508. At 5 bps: -4481. At 10 bps: -6946. The total notional traded was approximately 49.3 million index points across 32,692 transactions.

## 11. Trade log sample

The raw trade log — every executed BUY and SELL, with timestamps and prices.

In [ ]:
trades = result.trades_df
print(f"Total trades: {len(trades):,}")
trades.head(20)

## 12. Holdings at any time

what was held at every second — sample from one day

In [ ]:
hl = result.day_results[0].holdings_log   # NIFTY Nov 1
print(f"Holdings log shape: {hl.shape}")
hl.head(20)

In [ ]:
from datetime import date

roll_ts = result.trades_df[result.trades_df["date"] == date(2022, 11, 1)]
roll_ts = roll_ts[roll_ts["underlier"] == "BANKNIFTY"]
ts_counts = roll_ts.groupby("timestamp").size()
print("Roll size distribution:")
print(ts_counts.value_counts().head())

## 13. Rolling Sharpe ratio

5-day rolling Sharpe per underlier — shows whether performance is stable or deteriorating over the month, rather than collapsing everything into one noisy 21-day estimate.

In [ ]:
metrics.rolling_sharpe(result, window=5)

In [ ]:
fig = metrics.plot_rolling_sharpe(result, window=5)
fig.savefig(RESULTS_DIR / "10_rolling_sharpe.png", dpi=150, bbox_inches="tight")

### Rolling 5-day Sharpe ratio

**What:** Sharpe ratio computed over a rolling 5-trading-day window.
Each point answers: "how did the strategy perform over the last 5 days?"
The first 4 dates are NaN — a window of 5 is needed before the first estimate.

**Observations:**

- The rolling Sharpe is **negative for the entire month** for both underliers.
  There was no sustained 5-day period where the strategy made money
  consistently enough to produce a positive risk-adjusted return.

- **BANKNIFTY** (blue) deteriorates gradually and steadily through November,
  reaching its worst point around Nov 23–24 (rolling Sharpe ≈ −48).
  This reflects the compounding of consistent daily losses with little variation.

- **NIFTY** (orange) stays relatively stable at −5 to −10 through mid-November —
  meaning it was losing money but consistently, with low day-to-day variance.
  Then it **crashes sharply to −59 around Nov 24–25** before recovering.
  This spike corresponds to NIFTY's worst 5-day stretch of the month:
  Nov 21 (−33.70), Nov 22 (−28.85), Nov 23 (−27.15), Nov 24 (−15.25), Nov 25 (−32.90) —
  five consecutive losing days with high variance, which collapses the Sharpe.

- Both lines **partially recover toward month end** (back toward −20).
  This is not because the strategy started making money — it continued losing —
  but because the window rolled past the worst days and the remaining days
  had smaller, more consistent losses (lower variance improves Sharpe
  even when mean is still negative).